# EDA 07 — Nutrient Audit: Calories and Fat

This notebook consolidates the RecipeDB ingredient-level calorie/fat audit, unit standardisation diagnostics, macro-energy and mass-balance checks, and recipe-level validation.

## Inputs
- `dataset/raw/RecipeDB_full/recipe_ingredients.csv` (required)
- `dataset/raw/RecipeDB_full/recipe_nutrition.csv` (optional; recipe-level validation)

## Outputs
CSV files are written to `EDA/results/nutrient_audit/`.

## Critical interpretation
`energy_kcal` and `fat_g` are treated as recorded ingredient contributions. Dividing them by the recorded quantity is only a **conditional estimate** unless the source field semantics and quantity basis are confirmed. Therefore:
- This notebook does **not** create trusted nutrient values.
- Per-100g estimates are explicitly labelled `conditional_unverified`.
- No RecipeDB-derived profile should be used as a trusted substitution input.
- The notebook does not modify `ingredient_nutrients_final.csv` or any USDA mapping.
- Recipe-level agreement is a useful consistency diagnostic, but it does not prove every ingredient-level quantity/nutrient relationship.
- Ambiguous units (cups, spoons, pieces, etc.) are not converted to grams.
- No scaling, imputation, or automatic correction is performed.


In [1]:
from pathlib import Path
from fractions import Fraction
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Locate BTP_2 from the notebook working directory or its parents.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "dataset" / "raw" / "RecipeDB_full").is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate BTP_2. Run this notebook from BTP_2, EDA, or a child folder."
    )

RAW_DIR = PROJECT_ROOT / "dataset" / "raw" / "RecipeDB_full"
OUTPUT_DIR = PROJECT_ROOT / "EDA" / "results" / "nutrient_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INGREDIENT_FILE = RAW_DIR / "recipe_ingredients.csv"
NUTRITION_FILE = RAW_DIR / "recipe_nutrition.csv"

if not INGREDIENT_FILE.is_file():
    raise FileNotFoundError(f"Required input not found: {INGREDIENT_FILE}")

print("Project root :", PROJECT_ROOT)
print("Input folder :", RAW_DIR)
print("Output folder:", OUTPUT_DIR)
print("Recipe nutrition available:", NUTRITION_FILE.is_file())


Project root : /Users/harshgupta/Coding/BTP_2
Input folder : /Users/harshgupta/Coding/BTP_2/dataset/raw/RecipeDB_full
Output folder: /Users/harshgupta/Coding/BTP_2/EDA/results/nutrient_audit
Recipe nutrition available: True


## 1. Load source files and inspect their schema

Quantity parsing is deliberately more careful than `pd.to_numeric`: values such as `1/2` and `1 1/2` can be parsed, while ranges and ambiguous strings remain missing. This avoids silently losing valid fractional quantities or guessing at ranges.

In [2]:
ingredients_raw = pd.read_csv(INGREDIENT_FILE, low_memory=False)
ingredients_raw.columns = ingredients_raw.columns.astype(str).str.strip()

nutrition_raw = None
if NUTRITION_FILE.is_file():
    nutrition_raw = pd.read_csv(NUTRITION_FILE, low_memory=False)
    nutrition_raw.columns = nutrition_raw.columns.astype(str).str.strip()

print("Ingredient source shape:", ingredients_raw.shape)
print("Ingredient source columns:", ingredients_raw.columns.tolist())
display(ingredients_raw.head())

if nutrition_raw is not None:
    print("\nRecipe nutrition source shape:", nutrition_raw.shape)
    print("Recipe nutrition columns:", nutrition_raw.columns.tolist())
    display(nutrition_raw.head())


Ingredient source shape: (1191643, 10)
Ingredient source columns: ['recipe_id', 'recipe_title', 'ingredient_name', 'quantity', 'unit', 'state', 'energy_kcal', 'carbohydrates', 'protein_g', 'fat_g']


,recipe_id,recipe_title,ingredient_name,quantity,unit,state,energy_kcal,carbohydrates,protein_g,fat_g
0,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,tomato,4,NaN,ripe,167.44,37.128,8.736,1.456
1,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,chevre cheese cream,5,ounces,soft cut,126.0833,0.0,28.05,0.7083
2,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,pistachio,1/2,cup,shelled roasted chopped,344.4,16.7096,12.3984,27.8718
3,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,arugula,4,cups,NaN,20.0,2.92,2.064,0.528
4,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,extra virgin olive oil,1/2,cup,NaN,126.0833,0.0,28.05,0.7083



Recipe nutrition source shape: (18196640, 4)
Recipe nutrition columns: ['recipe_id', 'recipe_title', 'nutrient', 'quantity']


,recipe_id,recipe_title,nutrient,quantity
0,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,Total fats (g),30.5773
1,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,Energy (kCal),659.2041
2,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,Carbohydrates (g),57.1692
3,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,Protein (g),51.297
4,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,"Fatty acids, total saturated (g)",4.0870


## 2. Configuration and helper functions

The recorded nutrient values are audited separately from derived per-100g estimates. The plausibility ceilings below are review thresholds, not repair rules.

In [3]:
# Audit thresholds
MAX_RECORDED_KCAL = 5000.0
MAX_KCAL_PER_100G_REVIEW = 950.0
MAX_FAT_PER_100G_REVIEW = 100.0
MACRO_ABS_DIFF_KCAL = 50.0
MACRO_REL_DIFF = 0.25
RECIPE_KCAL_ABS_DIFF = 100.0
RECIPE_REL_DIFF = 0.25
RECIPE_FAT_ABS_DIFF_G = 5.0
RECIPE_FAT_REL_DIFF = 0.25
DUPLICATE_KCAL_TOLERANCE = 1.0
DUPLICATE_FAT_TOLERANCE_G = 0.1

MASS_TO_GRAMS = {
    "g": 1.0, "gram": 1.0, "grams": 1.0, "gm": 1.0, "gms": 1.0,
    "kg": 1000.0, "kgs": 1000.0, "kilogram": 1000.0, "kilograms": 1000.0,
    "mg": 0.001, "milligram": 0.001, "milligrams": 0.001,
    "ug": 0.000001, "mcg": 0.000001, "microgram": 0.000001,
    "micrograms": 0.000001, "µg": 0.000001,
    "oz": 28.349523125, "ounce": 28.349523125, "ounces": 28.349523125,
    "lb": 453.59237, "lbs": 453.59237, "pound": 453.59237, "pounds": 453.59237,
}

def find_column(frame, candidates):
    lookup = {str(c).strip().casefold(): c for c in frame.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    return None

def normalize_text(series):
    return (series.astype("string").str.strip().str.casefold()
            .str.replace(r"\s+", " ", regex=True))

def normalize_recipe_id(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def parse_quantity(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s or "-" in s or "–" in s or "to" in s.casefold():
        return np.nan
    try:
        return float(sum(Fraction(part) for part in s.split()))
    except (ValueError, ZeroDivisionError):
        try:
            return float(s)
        except (ValueError, TypeError):
            return np.nan

def normalize_unit(value):
    if pd.isna(value):
        return pd.NA
    unit = str(value).strip().casefold().replace(".", "").replace("_", " ")
    unit = re.sub(r"\s+", " ", unit).strip()
    aliases = {
        "gramme": "gram", "grammes": "grams", "kgs": "kg",
        "milligramme": "milligram", "milligrammes": "milligrams",
        "ounces (oz)": "oz", "pounds (lb)": "lb",
    }
    return aliases.get(unit, unit)

def write_csv(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved {filename}: {len(frame):,} rows")
    return path

def safe_spearman(frame, x, y):
    valid = frame[[x, y]].dropna()
    if len(valid) < 2:
        return np.nan
    return valid[x].corr(valid[y], method="spearman")


## 3. Validate required columns and prepare records

The raw quantity is preserved in `quantity_raw`; the parsed numeric value is stored separately. The original nutrient values are not changed apart from safe numeric parsing in the working copy.

In [4]:
required_columns = ["recipe_id", "ingredient_name", "quantity", "unit", "energy_kcal", "fat_g"]
missing = [c for c in required_columns if c not in ingredients_raw.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\nAvailable columns: {ingredients_raw.columns.tolist()}"
    )

df = ingredients_raw.copy()
df.insert(0, "source_row_number", np.arange(2, len(df) + 2))
df["quantity_raw"] = df["quantity"]

for col in ["ingredient_name", "unit", "recipe_title", "state"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df.loc[df[col].eq(""), col] = pd.NA

df["quantity_parsed"] = df["quantity_raw"].map(parse_quantity)
for col in ["energy_kcal", "fat_g", "carbohydrates", "protein_g"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["recipe_key"] = normalize_recipe_id(df["recipe_id"])
df["ingredient_key"] = normalize_text(df["ingredient_name"])
df["unit_normalized"] = df["unit"].map(normalize_unit)
df["mass_to_grams_factor"] = df["unit_normalized"].map(MASS_TO_GRAMS)
df["quantity_g"] = df["quantity_parsed"] * df["mass_to_grams_factor"]
df["mass_unit_standardized_flag"] = (
    df["mass_to_grams_factor"].notna() & df["quantity_g"].gt(0)
)

print(f"Ingredient rows: {len(df):,}")
print(f"Unique recipes: {df['recipe_key'].nunique(dropna=True):,}")
print(f"Unique ingredient names: {df['ingredient_key'].nunique(dropna=True):,}")
print(f"Quantities parsed: {df['quantity_parsed'].notna().sum():,}")
print(f"Rows with unambiguous mass conversion: {df['mass_unit_standardized_flag'].sum():,}")
display(df[["recipe_id", "ingredient_name", "quantity_raw", "quantity_parsed",
            "unit", "quantity_g", "energy_kcal", "fat_g"]].head(10))


Ingredient rows: 1,191,643
Unique recipes: 118,101
Unique ingredient names: 20,281
Quantities parsed: 1,083,149
Rows with unambiguous mass conversion: 69,370


,recipe_id,ingredient_name,quantity_raw,quantity_parsed,unit,quantity_g,energy_kcal,fat_g
0,100002,tomato,4,4.0,<NA>,NaN,167.4400,1.4560
1,100002,chevre cheese cream,5,5.0,ounces,141.747616,126.0833,0.7083
2,100002,pistachio,1/2,0.5,cup,NaN,344.4000,27.8718
3,100002,arugula,4,4.0,cups,NaN,20.0000,0.5280
4,100002,extra virgin olive oil,1/2,0.5,cup,NaN,126.0833,0.7083
5,100002,lemon,1,1.0,<NA>,NaN,1.2808,0.0132
6,100002,balsamic vinegar,1 -4,NaN,tablespoon,NaN,0.0000,0.0000
7,100002,salt,NaN,NaN,<NA>,NaN,NaN,NaN
8,100002,black pepper,NaN,NaN,<NA>,NaN,NaN,NaN
9,100002,cherry tomato,2,2.0,grape,NaN,126.0833,0.7083


## 4. Record-level quality audit

Missing values, zero/negative values, extreme values, duplicate rows, and repeated recipe/ingredient pairs are reported. Repeated ingredient entries are not automatically deleted because they may be legitimate recipe records.

In [5]:
df["missing_recipe_id_flag"] = df["recipe_key"].isna()
df["missing_ingredient_name_flag"] = df["ingredient_key"].isna() | df["ingredient_key"].eq("")
df["missing_quantity_flag"] = df["quantity_parsed"].isna()
df["nonpositive_quantity_flag"] = df["quantity_parsed"].le(0).fillna(False)
df["missing_unit_flag"] = df["unit_normalized"].isna() | df["unit_normalized"].eq("")

df["kcal_missing_flag"] = df["energy_kcal"].isna()
df["kcal_negative_flag"] = df["energy_kcal"].lt(0).fillna(False)
df["kcal_zero_flag"] = df["energy_kcal"].eq(0).fillna(False)
df["kcal_extreme_flag"] = df["energy_kcal"].gt(MAX_RECORDED_KCAL).fillna(False)

df["fat_missing_flag"] = df["fat_g"].isna()
df["fat_negative_flag"] = df["fat_g"].lt(0).fillna(False)
df["fat_zero_flag"] = df["fat_g"].eq(0).fillna(False)

def q3_plus_3iqr(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    q1, q3 = s.quantile(.25), s.quantile(.75)
    return q3 + 3 * (q3 - q1)

kcal_iqr_threshold = q3_plus_3iqr(df["energy_kcal"])
fat_iqr_threshold = q3_plus_3iqr(df["fat_g"])
df["kcal_high_iqr_flag"] = df["energy_kcal"].gt(kcal_iqr_threshold) if pd.notna(kcal_iqr_threshold) else False
df["fat_high_iqr_flag"] = df["fat_g"].gt(fat_iqr_threshold) if pd.notna(fat_iqr_threshold) else False

source_columns = list(ingredients_raw.columns)
df["exact_duplicate_row_flag"] = df.duplicated(subset=source_columns, keep="first")
pair_key = df["recipe_key"].astype("string") + "||" + df["ingredient_key"].fillna("")
df["repeated_recipe_ingredient_flag"] = (
    pair_key.duplicated(keep=False) & df["ingredient_key"].notna()
)

base_stats = pd.DataFrame([
    ["Ingredient rows", len(df)],
    ["Unique recipes", df["recipe_key"].nunique(dropna=True)],
    ["Unique ingredient names", df["ingredient_key"].nunique(dropna=True)],
    ["Missing kcal", int(df["kcal_missing_flag"].sum())],
    ["Zero kcal", int(df["kcal_zero_flag"].sum())],
    ["Negative kcal", int(df["kcal_negative_flag"].sum())],
    [f"Kcal > {MAX_RECORDED_KCAL:g}", int(df["kcal_extreme_flag"].sum())],
    ["Missing fat", int(df["fat_missing_flag"].sum())],
    ["Zero fat", int(df["fat_zero_flag"].sum())],
    ["Negative fat", int(df["fat_negative_flag"].sum())],
    ["Missing/unparseable quantity", int(df["missing_quantity_flag"].sum())],
    ["Nonpositive quantity", int(df["nonpositive_quantity_flag"].sum())],
    ["Rows with mass-standardised quantity", int(df["mass_unit_standardized_flag"].sum())],
    ["Exact duplicate rows beyond first", int(df["exact_duplicate_row_flag"].sum())],
    ["Rows in repeated recipe/ingredient groups", int(df["repeated_recipe_ingredient_flag"].sum())],
    ["Rows above kcal Q3 + 3*IQR", int(df["kcal_high_iqr_flag"].sum())],
    ["Rows above fat Q3 + 3*IQR", int(df["fat_high_iqr_flag"].sum())],
], columns=["Statistic", "Value"])
display(base_stats)

quantiles = [0, .01, .25, .50, .75, .90, .95, .99, 1]
distribution = pd.DataFrame({
    "quantile": quantiles,
    "energy_kcal": df["energy_kcal"].quantile(quantiles).to_numpy(),
    "fat_g": df["fat_g"].quantile(quantiles).to_numpy(),
})
display(distribution)

unit_summary = (
    df.assign(unit_label=df["unit_normalized"].fillna("[MISSING/UNSPECIFIED]"))
      .groupby("unit_label", dropna=False)
      .agg(
          rows=("recipe_id", "size"),
          unique_recipes=("recipe_key", "nunique"),
          unique_ingredients=("ingredient_key", "nunique"),
          kcal_missing=("energy_kcal", lambda s: int(s.isna().sum())),
          fat_missing=("fat_g", lambda s: int(s.isna().sum())),
          median_quantity=("quantity_parsed", "median"),
      ).reset_index().sort_values("rows", ascending=False)
)
display(unit_summary.head(30))
write_csv(unit_summary, "nutrient_kcal_fat_unit_summary.csv")


,Statistic,Value
0,Ingredient rows,1191643
1,Unique recipes,118101
2,Unique ingredient names,20281
3,Missing kcal,400143
4,Zero kcal,46048
5,Negative kcal,0
6,Kcal > 5000,13376
7,Missing fat,400143
8,Zero fat,99372
9,Negative fat,0


,quantile,energy_kcal,fat_g
0,0.00,0.000000e+00,0.000000
1,0.01,0.000000e+00,0.000000
2,0.25,7.845175e+00,0.034700
3,0.50,6.400000e+01,0.403200
4,0.75,2.860000e+02,7.510600
5,0.90,8.275862e+02,40.500000
6,0.95,1.548000e+03,83.914700
7,0.99,1.320404e+04,465.595055
8,1.00,2.989800e+06,300542.850000


,unit_label,rows,unique_recipes,unique_ingredients,kcal_missing,fat_missing,median_quantity
74,[MISSING/UNSPECIFIED],333596,103175,7401,157870,157870,2.00
311,cup,207349,85064,5546,51065,51065,0.50
968,teaspoon,158745,69506,2015,55544,55544,0.50
928,tablespoons,107121,63254,3078,19923,19923,2.00
922,tablespoon,70969,45933,2234,14550,14550,1.00
324,cups,68967,47385,2984,19532,19532,2.00
971,teaspoons,35857,26326,1501,9865,9865,2.00
603,ounces,33193,21827,3280,172,172,8.00
175,can,25950,18137,1164,7637,7637,1.00
8,1/2,18840,15453,1990,6334,6334,1.50


Saved nutrient_kcal_fat_unit_summary.csv: 1,081 rows


PosixPath('/Users/harshgupta/Coding/BTP_2/EDA/results/nutrient_audit/nutrient_kcal_fat_unit_summary.csv')

## 5. Macro-energy and mass-balance diagnostics

Where all three macros exist, approximate energy is calculated as `4 × carbohydrates + 4 × protein + 9 × fat`. A row is flagged only when both the absolute difference is greater than 50 kcal and relative difference is greater than 25%.

A separate mass-balance flag checks whether the sum of carbohydrate, protein, and fat grams exceeds the ingredient's recorded gram quantity by more than 5%. This is a diagnostic for potential unit/field-basis mismatch, not proof of error.

In [6]:
macro_cols = ["carbohydrates", "protein_g", "fat_g"]
if all(c in df.columns for c in macro_cols):
    macros_present = df[macro_cols].notna().all(axis=1)
    df["macro_implied_kcal"] = 4*df["carbohydrates"] + 4*df["protein_g"] + 9*df["fat_g"]
    df["macro_energy_abs_difference_kcal"] = (df["energy_kcal"] - df["macro_implied_kcal"]).abs()
    denom = pd.concat([df["energy_kcal"].abs(), df["macro_implied_kcal"].abs()], axis=1).max(axis=1).replace(0, np.nan)
    df["macro_energy_relative_difference"] = df["macro_energy_abs_difference_kcal"] / denom
    df["macro_energy_review_flag"] = (
        df["energy_kcal"].notna() & macros_present
        & df["macro_energy_abs_difference_kcal"].gt(MACRO_ABS_DIFF_KCAL)
        & df["macro_energy_relative_difference"].gt(MACRO_REL_DIFF)
    )
    df["macro_mass_g"] = df["carbohydrates"] + df["protein_g"] + df["fat_g"]
    df["macro_mass_fraction"] = df["macro_mass_g"] / df["quantity_g"].replace(0, np.nan)
    df["macro_mass_balance_review_flag"] = (
        df["mass_unit_standardized_flag"] & macros_present
        & df["macro_mass_fraction"].gt(1.05)
    )
else:
    df["macro_implied_kcal"] = np.nan
    df["macro_energy_abs_difference_kcal"] = np.nan
    df["macro_energy_relative_difference"] = np.nan
    df["macro_energy_review_flag"] = False
    df["macro_mass_g"] = np.nan
    df["macro_mass_fraction"] = np.nan
    df["macro_mass_balance_review_flag"] = False
    print("Macro columns incomplete: macro-energy and mass-balance checks skipped.")

macro_cols_out = [c for c in [
    "source_row_number", "recipe_id", "recipe_title", "ingredient_name",
    "quantity_raw", "quantity_parsed", "unit", "quantity_g",
    "energy_kcal", "carbohydrates", "protein_g", "fat_g",
    "macro_implied_kcal", "macro_energy_abs_difference_kcal",
    "macro_energy_relative_difference", "macro_energy_review_flag",
    "macro_mass_g", "macro_mass_fraction", "macro_mass_balance_review_flag"
] if c in df.columns]

macro_review = df.loc[
    df["energy_kcal"].notna() & df["macro_implied_kcal"].notna(),
    macro_cols_out
].copy()
print("Rows with kcal and all macros:", f"{len(macro_review):,}")
print("Macro-energy disagreement rows:", f"{int(df['macro_energy_review_flag'].sum()):,}")
print("Macro mass-balance review rows:", f"{int(df['macro_mass_balance_review_flag'].sum()):,}")
display(macro_review.head())
write_csv(macro_review, "nutrient_kcal_fat_macro_energy_consistency.csv")


Rows with kcal and all macros: 791,500
Macro-energy disagreement rows: 2,782
Macro mass-balance review rows: 452


,source_row_number,recipe_id,recipe_title,ingredient_name,quantity_raw,quantity_parsed,unit,quantity_g,energy_kcal,carbohydrates,protein_g,fat_g,macro_implied_kcal,macro_energy_abs_difference_kcal,macro_energy_relative_difference,macro_energy_review_flag,macro_mass_g,macro_mass_fraction,macro_mass_balance_review_flag
0,2,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,tomato,4,4.0,<NA>,NaN,167.4400,37.1280,8.7360,1.4560,196.5600,29.1200,0.148148,False,47.3200,NaN,False
1,3,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,chevre cheese cream,5,5.0,ounces,141.747616,126.0833,0.0000,28.0500,0.7083,118.5747,7.5086,0.059553,False,28.7583,0.202884,False
2,4,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,pistachio,1/2,0.5,cup,NaN,344.4000,16.7096,12.3984,27.8718,367.2782,22.8782,0.062291,False,56.9798,NaN,False
3,5,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,arugula,4,4.0,cups,NaN,20.0000,2.9200,2.0640,0.5280,24.6880,4.6880,0.189890,False,5.5120,NaN,False
4,6,100002,Heirloom Tomato Salad With Goat Cheese and Aru...,extra virgin olive oil,1/2,0.5,cup,NaN,126.0833,0.0000,28.0500,0.7083,118.5747,7.5086,0.059553,False,28.7583,NaN,False


Saved nutrient_kcal_fat_macro_energy_consistency.csv: 791,500 rows


PosixPath('/Users/harshgupta/Coding/BTP_2/EDA/results/nutrient_audit/nutrient_kcal_fat_macro_energy_consistency.csv')

## 6. Conditional per-100g diagnostics — not trusted nutrient profiles

The calculations below are retained only to inspect the behaviour of the RecipeDB fields:

- `conditional_kcal_per_100g = energy_kcal / quantity_g × 100`
- `conditional_fat_per_100g = fat_g / quantity_g × 100`

Even when the unit converts cleanly to grams, these values remain conditional because the meaning/basis of `energy_kcal`, `fat_g`, and `quantity` has not been proven to match at ingredient level. The output explicitly records this status. No `trusted` tier is assigned.

In [7]:
valid_mass = df["mass_unit_standardized_flag"]

df["conditional_kcal_per_100g"] = np.where(
    valid_mass & df["energy_kcal"].notna() & df["energy_kcal"].ge(0),
    df["energy_kcal"] / df["quantity_g"] * 100,
    np.nan
)
df["conditional_fat_per_100g"] = np.where(
    valid_mass & df["fat_g"].notna() & df["fat_g"].ge(0),
    df["fat_g"] / df["quantity_g"] * 100,
    np.nan
)

df["conditional_kcal_density_review_flag"] = (
    df["conditional_kcal_per_100g"].notna()
    & df["conditional_kcal_per_100g"].gt(MAX_KCAL_PER_100G_REVIEW)
)
df["conditional_fat_density_review_flag"] = (
    df["conditional_fat_per_100g"].notna()
    & df["conditional_fat_per_100g"].gt(MAX_FAT_PER_100G_REVIEW)
)

# A diagnostic eligibility mask: removes clearly invalid/flagged records,
# but does NOT make remaining estimates trusted.
common_diag = (
    df["ingredient_key"].notna() & df["ingredient_key"].ne("")
    & df["recipe_key"].notna()
    & valid_mass
    & ~df["exact_duplicate_row_flag"]
    & ~df["nonpositive_quantity_flag"]
    & ~df["macro_mass_balance_review_flag"]
)
kcal_diag_eligible = (
    common_diag & df["energy_kcal"].notna()
    & ~df["kcal_negative_flag"] & ~df["kcal_zero_flag"]
    & ~df["kcal_extreme_flag"]
    & ~df["macro_energy_review_flag"]
    & ~df["conditional_kcal_density_review_flag"]
    & df["conditional_kcal_per_100g"].notna()
)
fat_diag_eligible = (
    common_diag & df["fat_g"].notna()
    & ~df["fat_negative_flag"]
    & ~df["conditional_fat_density_review_flag"]
    & df["conditional_fat_per_100g"].notna()
)

print("Rows with conditional kcal/100g:", f"{df['conditional_kcal_per_100g'].notna().sum():,}")
print("Rows with conditional fat/100g:", f"{df['conditional_fat_per_100g'].notna().sum():,}")
print("Kcal diagnostic eligible rows:", f"{kcal_diag_eligible.sum():,}")
print("Fat diagnostic eligible rows:", f"{fat_diag_eligible.sum():,}")
print("Kcal density review rows:", f"{df['conditional_kcal_density_review_flag'].sum():,}")
print("Fat density review rows:", f"{df['conditional_fat_density_review_flag'].sum():,}")


Rows with conditional kcal/100g: 68,928
Rows with conditional fat/100g: 68,928
Kcal diagnostic eligible rows: 66,561
Fat diagnostic eligible rows: 67,810
Kcal density review rows: 290
Fat density review rows: 209


## 7. Recipe-level validation (diagnostic only)

The notebook compares the sum of ingredient-level recorded nutrient contributions with recipe-level `recipe_nutrition.csv` values, if available.

- Calories: only exact `Energy (kcal)` labels are selected; `Energy (kJ)` is excluded.
- Fat: only explicit total-fat gram labels are selected; saturated fat is not treated as total fat.
- Duplicate recipe-level nutrient rows are summarised by their median, with conflicting values flagged.
- No factor is applied and no source value is modified.

In [8]:
def recipe_validation(nutrition_frame, nutrient):
    if nutrition_frame is None:
        print(f"Skipping recipe-level {nutrient} validation: recipe_nutrition.csv unavailable.")
        return None

    rid = find_column(nutrition_frame, ["recipe_id", "recipeid", "recipe id"])
    label_col = find_column(nutrition_frame, ["nutrient", "nutrient_name", "name"])
    value_col = find_column(nutrition_frame, ["quantity", "value", "amount", "nutrient_quantity"])
    if not all([rid, label_col, value_col]):
        print(f"Skipping {nutrient}: required columns not identified.")
        return None

    tmp = nutrition_frame.copy()
    tmp["_recipe_key"] = normalize_recipe_id(tmp[rid])
    tmp["_label"] = tmp[label_col].astype("string").str.strip()
    tmp["_label_norm"] = tmp["_label"].str.casefold().str.replace(r"\s+", " ", regex=True)
    tmp["_value"] = pd.to_numeric(tmp[value_col], errors="coerce")

    if nutrient == "kcal":
        mask = tmp["_label_norm"].str.fullmatch(r"energy\s*\(\s*kcal\s*\)", na=False)
        ing_col = "energy_kcal"
        tolerance = DUPLICATE_KCAL_TOLERANCE
        abs_threshold = RECIPE_KCAL_ABS_DIFF
    else:
        mask = tmp["_label_norm"].str.fullmatch(
            r"(?:total\s+)?fat\s*\(\s*g\s*\)|total lipid \(fat\) \(g\)",
            na=False
        )
        ing_col = "fat_g"
        tolerance = DUPLICATE_FAT_TOLERANCE_G
        abs_threshold = RECIPE_FAT_ABS_DIFF_G

    selected = tmp.loc[mask].copy()
    if selected.empty:
        print(f"No unambiguous recipe-level {nutrient} labels found; validation skipped.")
        print("Example labels:", tmp["_label"].dropna().drop_duplicates().head(30).tolist())
        return None

    recipe_agg = (
        selected.groupby("_recipe_key", dropna=False)
        .agg(
            recipe_nutrient_rows=("_value", "size"),
            recipe_nutrient_nonmissing=("_value", "count"),
            recipe_nutrient_value=("_value", "median"),
            recipe_nutrient_min=("_value", "min"),
            recipe_nutrient_max=("_value", "max"),
        ).reset_index()
    )
    recipe_agg["multiple_nutrient_rows_flag"] = recipe_agg["recipe_nutrient_rows"].gt(1)
    recipe_agg["conflicting_nutrient_rows_flag"] = (
        recipe_agg["recipe_nutrient_max"] - recipe_agg["recipe_nutrient_min"]
    ).gt(tolerance)

    ingredient_agg = (
        df.groupby("recipe_key", dropna=False)
        .agg(
            ingredient_rows=(ing_col, "size"),
            ingredient_nutrient_nonmissing=(ing_col, "count"),
            ingredient_nutrient_sum=(ing_col, "sum"),
        ).reset_index().rename(columns={"recipe_key": "_recipe_key"})
    )
    ingredient_agg.loc[
        ingredient_agg["ingredient_nutrient_nonmissing"].eq(0),
        "ingredient_nutrient_sum"
    ] = np.nan

    result = ingredient_agg.merge(recipe_agg, on="_recipe_key", how="outer")
    result["ingredient_minus_recipe"] = result["ingredient_nutrient_sum"] - result["recipe_nutrient_value"]
    result["absolute_difference"] = result["ingredient_minus_recipe"].abs()
    result["ingredient_to_recipe_ratio"] = (
        result["ingredient_nutrient_sum"] / result["recipe_nutrient_value"].replace(0, np.nan)
    )
    result["absolute_relative_difference"] = (
        result["absolute_difference"] / result["recipe_nutrient_value"].abs().replace(0, np.nan)
    )
    both = result["ingredient_nutrient_sum"].notna() & result["recipe_nutrient_value"].notna()
    result["large_difference_flag"] = (
        both
        & result["absolute_difference"].gt(abs_threshold)
        & result["absolute_relative_difference"].gt(RECIPE_REL_DIFF)
    )
    result["nutrient"] = nutrient

    print(f"\n{nutrient.upper()} recipe validation")
    print("Selected recipe nutrient rows:", f"{len(selected):,}")
    print("Recipes compared:", f"{int(both.sum()):,}")
    print("Multiple nutrient rows:", f"{int(result['multiple_nutrient_rows_flag'].sum()):,}")
    print("Conflicting nutrient rows:", f"{int(result['conflicting_nutrient_rows_flag'].sum()):,}")
    print("Large-difference flags:", f"{int(result['large_difference_flag'].sum()):,}")
    print("Spearman correlation:", round(safe_spearman(result.loc[both], "ingredient_nutrient_sum", "recipe_nutrient_value"), 4)
          if both.sum() >= 2 else np.nan)
    print("Median ingredient/recipe ratio:", result.loc[both, "ingredient_to_recipe_ratio"].median())
    return result

kcal_recipe_comparison = recipe_validation(nutrition_raw, "kcal")
fat_recipe_comparison = recipe_validation(nutrition_raw, "fat")

if kcal_recipe_comparison is not None:
    write_csv(kcal_recipe_comparison, "nutrient_kcal_recipe_comparison.csv")
    write_csv(kcal_recipe_comparison[kcal_recipe_comparison["large_difference_flag"]],
              "nutrient_kcal_recipe_large_differences.csv")
if fat_recipe_comparison is not None:
    write_csv(fat_recipe_comparison, "nutrient_fat_recipe_comparison.csv")
    write_csv(fat_recipe_comparison[fat_recipe_comparison["large_difference_flag"]],
              "nutrient_fat_recipe_large_differences.csv")



KCAL recipe validation
Selected recipe nutrient rows: 236,320
Recipes compared: 117,030
Multiple nutrient rows: 118,160
Conflicting nutrient rows: 0
Large-difference flags: 15,082
Spearman correlation: 0.972
Median ingredient/recipe ratio: 1.0

FAT recipe validation
Selected recipe nutrient rows: 118,160
Recipes compared: 117,030
Multiple nutrient rows: 0
Conflicting nutrient rows: 0
Large-difference flags: 8,073
Spearman correlation: 0.9919
Median ingredient/recipe ratio: 1.0
Saved nutrient_kcal_recipe_comparison.csv: 118,160 rows
Saved nutrient_kcal_recipe_large_differences.csv: 15,082 rows
Saved nutrient_fat_recipe_comparison.csv: 118,160 rows
Saved nutrient_fat_recipe_large_differences.csv: 8,073 rows


## 8. Ingredient-level conditional profiles

These summaries describe the distribution of the conditional per-100g estimates across records. They are **not trusted nutrient profiles** and are not intended to be consumed directly by the substitution pipeline. The profile status is always `CONDITIONAL_UNVERIFIED_REVIEW_ONLY`.

In [9]:
def conditional_profile(data, value_col, eligible, nutrient):
    x = data.loc[eligible].copy()
    if x.empty:
        return pd.DataFrame(columns=[
            "ingredient_key", "ingredient_name", "observation_count", "recipe_count",
            "median_per_100g", "mean_per_100g", "q25_per_100g", "q75_per_100g",
            "iqr_per_100g", "profile_status"
        ])
    out = (
        x.groupby("ingredient_key", dropna=False)
         .agg(
             ingredient_name=("ingredient_name", "first"),
             observation_count=(value_col, "count"),
             recipe_count=("recipe_key", "nunique"),
             median_per_100g=(value_col, "median"),
             mean_per_100g=(value_col, "mean"),
             q25_per_100g=(value_col, lambda s: s.quantile(.25)),
             q75_per_100g=(value_col, lambda s: s.quantile(.75)),
         ).reset_index()
    )
    out["iqr_per_100g"] = out["q75_per_100g"] - out["q25_per_100g"]
    out["nutrient"] = nutrient
    out["profile_status"] = "CONDITIONAL_UNVERIFIED_REVIEW_ONLY"
    out["warning"] = (
        "Derived from RecipeDB ingredient contribution divided by recorded quantity; "
        "not approved as a trusted per-100g substitution value."
    )
    return out

kcal_profiles_review = conditional_profile(
    df, "conditional_kcal_per_100g", kcal_diag_eligible, "kcal"
)
fat_profiles_review = conditional_profile(
    df, "conditional_fat_per_100g", fat_diag_eligible, "fat"
)

print("Conditional calorie profiles:", f"{len(kcal_profiles_review):,}")
print("Conditional fat profiles:", f"{len(fat_profiles_review):,}")
display(kcal_profiles_review.head(10))
display(fat_profiles_review.head(10))

write_csv(kcal_profiles_review, "nutrient_kcal_conditional_profiles_review_only.csv")
write_csv(fat_profiles_review, "nutrient_fat_conditional_profiles_review_only.csv")


Conditional calorie profiles: 5,530
Conditional fat profiles: 5,566


,ingredient_key,ingredient_name,observation_count,recipe_count,median_per_100g,mean_per_100g,q25_per_100g,q75_per_100g,iqr_per_100g,nutrient,profile_status,warning
0,'crunchie chocolate candy bar,'crunchie chocolate candy bar,1,1,92.708333,92.708333,92.708333,92.708333,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
1,1lb,1lb,1,1,268.000000,268.000000,268.000000,268.000000,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
2,5oz gluten flour,5oz gluten flour,1,1,89.000000,89.000000,89.000000,89.000000,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
3,abalone without shell,abalone without shell,2,2,88.949380,88.949380,88.949380,88.949380,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
4,aberdeen,aberdeen,1,1,88.949181,88.949181,88.949181,88.949181,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
5,aberdeen sirloin steak,aberdeen sirloin steak,1,1,88.949380,88.949380,88.949380,88.949380,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
6,absinthe,absinthe,3,3,88.949292,88.949233,88.949203,88.949292,0.000088,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
7,absolut citron vodka,absolut citron vodka,1,1,88.949115,88.949115,88.949115,88.949115,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
8,absolut kurant vodka,absolut kurant vodka,1,1,266.847522,266.847522,266.847522,266.847522,0.000000,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
9,absolut vodka,absolut vodka,2,2,88.949203,88.949203,88.949159,88.949248,0.000088,kcal,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...


,ingredient_key,ingredient_name,observation_count,recipe_count,median_per_100g,mean_per_100g,q25_per_100g,q75_per_100g,iqr_per_100g,nutrient,profile_status,warning
0,'crunchie chocolate candy bar,'crunchie chocolate candy bar,1,1,0.520833,0.520833,0.520833,0.520833,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
1,1lb,1lb,1,1,15.800000,15.800000,15.800000,15.800000,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
2,5oz gluten flour,5oz gluten flour,1,1,0.500000,0.500000,0.500000,0.500000,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
3,abalone without shell,abalone without shell,2,2,0.499705,0.499705,0.499703,0.499708,0.000006,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
4,aberdeen,aberdeen,1,1,0.499722,0.499722,0.499722,0.499722,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
5,aberdeen sirloin steak,aberdeen sirloin steak,1,1,0.499711,0.499711,0.499711,0.499711,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
6,absinthe,absinthe,3,3,0.499832,0.499773,0.499744,0.499832,0.000088,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
7,absolut citron vodka,absolut citron vodka,1,1,0.499656,0.499656,0.499656,0.499656,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
8,absolut kurant vodka,absolut kurant vodka,1,1,1.499143,1.499143,1.499143,1.499143,0.000000,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...
9,absolut vodka,absolut vodka,2,2,0.499744,0.499744,0.499700,0.499788,0.000088,fat,CONDITIONAL_UNVERIFIED_REVIEW_ONLY,Derived from RecipeDB ingredient contribution ...


Saved nutrient_kcal_conditional_profiles_review_only.csv: 5,530 rows
Saved nutrient_fat_conditional_profiles_review_only.csv: 5,566 rows


PosixPath('/Users/harshgupta/Coding/BTP_2/EDA/results/nutrient_audit/nutrient_fat_conditional_profiles_review_only.csv')

## 9. Save record-level audit and review outputs

The combined cleaned file preserves source values and adds audit/diagnostic columns. Separate eligible-row files are explicitly named `conditional` and `review_only` to prevent accidental use as trusted substitution inputs.

In [10]:
# Assemble all review flags without deleting or correcting any source record.
flag_columns = [
    "missing_recipe_id_flag", "missing_ingredient_name_flag",
    "missing_quantity_flag", "nonpositive_quantity_flag", "missing_unit_flag",
    "kcal_missing_flag", "kcal_negative_flag", "kcal_zero_flag", "kcal_extreme_flag",
    "fat_missing_flag", "fat_negative_flag", "fat_zero_flag",
    "kcal_high_iqr_flag", "fat_high_iqr_flag",
    "macro_energy_review_flag", "macro_mass_balance_review_flag",
    "conditional_kcal_density_review_flag", "conditional_fat_density_review_flag",
    "exact_duplicate_row_flag", "repeated_recipe_ingredient_flag"
]
df["quality_flags"] = ""
for col in flag_columns:
    df.loc[df[col].fillna(False), "quality_flags"] += col.replace("_flag", "").upper() + ";"
df["quality_flags"] = df["quality_flags"].str.rstrip(";")

write_csv(df, "nutrient_kcal_fat_cleaned_records.csv")
write_csv(df[df["quality_flags"].ne("")].copy(), "nutrient_kcal_fat_review_flags.csv")
write_csv(df[kcal_diag_eligible].copy(), "nutrient_kcal_conditional_rows_review_only.csv")
write_csv(df[fat_diag_eligible].copy(), "nutrient_fat_conditional_rows_review_only.csv")
write_csv(df[df["exact_duplicate_row_flag"]].copy(), "nutrient_kcal_fat_duplicate_rows.csv")
write_csv(df[df["repeated_recipe_ingredient_flag"]].copy(),
          "nutrient_kcal_fat_repeated_recipe_ingredients.csv")


Saved nutrient_kcal_fat_cleaned_records.csv: 1,191,643 rows
Saved nutrient_kcal_fat_review_flags.csv: 779,934 rows
Saved nutrient_kcal_conditional_rows_review_only.csv: 66,561 rows
Saved nutrient_fat_conditional_rows_review_only.csv: 67,810 rows
Saved nutrient_kcal_fat_duplicate_rows.csv: 6,852 rows
Saved nutrient_kcal_fat_repeated_recipe_ingredients.csv: 72,131 rows


PosixPath('/Users/harshgupta/Coding/BTP_2/EDA/results/nutrient_audit/nutrient_kcal_fat_repeated_recipe_ingredients.csv')

## 10. Final summary

The notebook summary is displayed here rather than saved as a separate text report. The output inventory is printed at the end.

In [11]:
summary = pd.DataFrame([
    ["Source ingredient rows", len(df)],
    ["Unique recipes", df["recipe_key"].nunique(dropna=True)],
    ["Unique ingredient names", df["ingredient_key"].nunique(dropna=True)],
    ["Rows with parsed quantity", int(df["quantity_parsed"].notna().sum())],
    ["Rows with reliable mass-unit conversion", int(df["mass_unit_standardized_flag"].sum())],
    ["Rows with conditional kcal/100g", int(df["conditional_kcal_per_100g"].notna().sum())],
    ["Rows with conditional fat/100g", int(df["conditional_fat_per_100g"].notna().sum())],
    ["Kcal diagnostic eligible rows (review only)", int(kcal_diag_eligible.sum())],
    ["Fat diagnostic eligible rows (review only)", int(fat_diag_eligible.sum())],
    ["Macro-energy disagreement rows", int(df["macro_energy_review_flag"].sum())],
    ["Macro mass-balance review rows", int(df["macro_mass_balance_review_flag"].sum())],
    ["Kcal density review rows", int(df["conditional_kcal_density_review_flag"].sum())],
    ["Fat density review rows", int(df["conditional_fat_density_review_flag"].sum())],
    ["Exact duplicate rows beyond first", int(df["exact_duplicate_row_flag"].sum())],
    ["Rows with one or more review flags", int(df["quality_flags"].ne("").sum())],
], columns=["Metric", "Value"])
display(summary)

print("\nOutput CSV files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {path.name} ({path.stat().st_size / (1024*1024):.2f} MB)")

print("\nIMPORTANT:")
print("1. RecipeDB-derived per-100g profiles are conditional and unverified.")
print("2. Do not use these review-only profiles as trusted substitution values.")
print("3. Use the USDA-based ingredient_nutrients_final.csv only after its mappings are reviewed.")
print("4. No raw source file, USDA mapping, or existing final nutrient table was modified.")


,Metric,Value
0,Source ingredient rows,1191643
1,Unique recipes,118101
2,Unique ingredient names,20281
3,Rows with parsed quantity,1083149
4,Rows with reliable mass-unit conversion,69370
5,Rows with conditional kcal/100g,68928
6,Rows with conditional fat/100g,68928
7,Kcal diagnostic eligible rows (review only),66561
8,Fat diagnostic eligible rows (review only),67810
9,Macro-energy disagreement rows,2782



Output CSV files:
- ingredient_nutrients_final.csv (0.07 MB)
- ingredient_per100g_with_tiers.csv (0.20 MB)
- nutrient_fat_conditional_profiles_review_only.csv (1.52 MB)
- nutrient_fat_conditional_rows_review_only.csv (25.64 MB)
- nutrient_fat_ingredient_profiles.csv (0.72 MB)
- nutrient_fat_recipe_comparison.csv (13.04 MB)
- nutrient_fat_recipe_large_differences.csv (1.09 MB)
- nutrient_fat_trusted_standardized_records.csv (19.98 MB)
- nutrient_kcal_conditional_profiles_review_only.csv (1.50 MB)
- nutrient_kcal_conditional_rows_review_only.csv (25.21 MB)
- nutrient_kcal_fat_cleaned_records.csv (345.96 MB)
- nutrient_kcal_fat_duplicate_rows.csv (2.25 MB)
- nutrient_kcal_fat_macro_energy_consistency.csv (123.78 MB)
- nutrient_kcal_fat_repeated_recipe_ingredients.csv (22.80 MB)
- nutrient_kcal_fat_review_flags.csv (218.58 MB)
- nutrient_kcal_fat_unit_summary.csv (0.02 MB)
- nutrient_kcal_ingredient_profiles.csv (0.70 MB)
- nutrient_kcal_recipe_comparison.csv (13.32 MB)
- nutrient_kcal_re